In [1]:
# Check for GPU access with PyTorch
import torch
torch.cuda.is_available()

# Setup device agnostic code
# set the device to use cuda if available, if not, use cpu
device = "cuda" if torch.cuda.is_available() else "cpu"
device

c:\Users\adria\anaconda3\envs\ECE4078\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

In [2]:
import h5py
import numpy as np
import pandas as pd
import csv
import array
import os
import re
import time
# for method 3
#import  jpype
#import  asposecells
#jpype.startJVM()
#from asposecells.api import Workbook

Converting h5 and csv to xlsx for all 150 subjects

In [7]:
root_directory = 'C:/Users/adria/OneDrive - Monash University/1 Raw Data'
# root_directory = 'C:/Users/adria/OneDrive - Monash University/MEngSc/Measurement System/1 Validation/Starting Position - Movement'

file_types = ['.h5', '_Joint_Angles.csv']
folders_to_exclude_main = ['Not Relevant', '1 DCS, ICF, VRAF']
folders_to_exclude = ['Not Relevant', '1 DCS, ICF, VRAF']
visited_dirs = []
excel_already_created = 0 # 0 means no, 1 means yes

for root_main, dirs_main, files_main in os.walk(root_directory, followlinks=False):

    dirs_main[:] = [d_main for d_main in dirs_main if d_main not in folders_to_exclude_main]

    for folder_name in dirs_main:  # Loop through each folder (i.e.: each subject) in the current directory
        
        start_time = time.time()

        folder_path = os.path.join(root_main, folder_name)  # Get the full path to the folder
        
        # Check if the folder is already visited or is a sub-path of any visited folder
        if any(folder_path.startswith(visited + os.sep) for visited in visited_dirs): # or re.search(r'_Incomplete_', folder_name):
            continue
        else:
            subject_code_match = re.search(r'(HC_\d+|PD_\d+|ET_\d+)', folder_name)
            if subject_code_match:
                subject_code = subject_code_match.group(0)  # Extract the matched pattern
                subject_code = subject_code.replace("_", " ")
                print(f"Subject Code: {subject_code}")
            else:
                continue
                # print("No valid subject code found in the file path.")
            subject_group_match = re.search(r'(_HC_+|_PD_+|_ET_+)', folder_name)
            if subject_group_match:
                subject_group = subject_group_match.group(0)  # Extract the matched pattern
                subject_group = subject_group.replace("_", "")
            else:
                continue
                # print("No valid subject group found in the file path.")

        # Check if the combined data excel file has already been created
        combined_data_file_path = f'C:/Users/adria/OneDrive - Monash University/2 Raw Combined Data/' #{subject_group}/{subject_code}.xlsx
        # if os.path.isfile(combined_data_file_path):
        #     print(f'{subject_code}.xlsx has already been created, proceeding with the next subject.\n')
        #     continue
        combined_excel_filename = f'{subject_code}.xlsx'
        for root, dirs, files in os.walk(combined_data_file_path):
            if combined_excel_filename in files:
                print(f'{subject_code}.xlsx has already been created, proceeding with the next subject.\n')
                excel_already_created = 1
                break
        
        if excel_already_created == 1:
            excel_already_created = 0
            continue

        # Path directory for the new excel file
        new_excel_folder_path = f'C:/Users/adria/OneDrive - Monash University/2 Raw Combined Data/{subject_group}/'
        # new_excel_folder_path = 'C:/Users/adria/OneDrive - Monash University/MEngSc/Measurement System/1 Validation/Starting Position - Movement/'

        # Create the output folder if it doesn't exist
        if not os.path.exists(new_excel_folder_path):
            os.makedirs(new_excel_folder_path)

        new_excel_file_path = new_excel_folder_path + f'{subject_code}.xlsx'
        
        # Add the current folder to the visited directories
        visited_dirs.append(folder_path)

        csv_all_dataframe, h5_all_dataframe, time_dataframe = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

        # To find the h5 file with the longest recordings
        time_array_length = 0
        longest_device_time_array = []
        longest_time_array = []
        for root, dirs, files in os.walk(folder_path, followlinks=False):
            dirs[:] = [d for d in dirs if d not in folders_to_exclude]

            for file_name in files:
                # Construct the absolute path of the file
                file_path = os.path.join(root, file_name)

                # Check if the file's extension matches any of the specified file types
                if (file_name.endswith('.h5') and re.search(r'\d+\.h5$', file_name)):

                    # Open the file and read its contents
                    dataset = h5py.File(file_path, 'r')

                    sensorslist = dataset['Sensors']
                    sensor_list_keys = list(sensorslist.keys())

                    sensor = sensorslist[sensor_list_keys[0]] # take any sensor will do since same recording all 6 sensors will have same length of recording

                    time_values = np.array(list(sensor['Time']))
                    #converted_time = array.array('f')
                    converted_time = np.array([None] * len(time_values))
                    for j in range(len(time_values)):
                        if j == 0:
                            time_zero = time_values[j]
                            converted_time[j] = 0
                        else:
                            converted_time[j] = (time_values[j] - time_zero)/(1000*1000)

                    if (len(time_values) > time_array_length):
                        longest_device_time_array = time_values
                        longest_time_array = converted_time
                        time_array_length = len(time_values)

        time_dataframe = pd.DataFrame({"Device Time (milliseconds)": longest_device_time_array, "Time": longest_time_array})
        h5_all_dataframe = pd.concat([h5_all_dataframe, time_dataframe], ignore_index = True, axis = 1)
        csv_all_dataframe = pd.concat([csv_all_dataframe, time_dataframe], ignore_index = True, axis = 1)

        file_number = 1
        # Walk through all the directories and files starting from the root directory
        for root, dirs, files in os.walk(folder_path, followlinks=False):
            dirs[:] = [d for d in dirs if d not in folders_to_exclude]

            for file_name in files:
                # Construct the absolute path of the file
                file_path = os.path.join(root, file_name)

                # Check if the file's extension matches any of the specified file types
                if (file_name.endswith('.h5') and re.search(r'\d+\.h5$', file_name)):

                    file_name_without_extension = os.path.splitext(file_name)[0]

                    # Get the final folder name in the root path
                    #final_folder_name = os.path.basename(root)

                    # print("File path:", file_path)
                    # print("Root:", root)
                    print(f'File {file_number}: {file_name}')
                    # print("Dir:", dirs)
                    # print("File name without extension:", file_name_without_extension)
                    # print("Files:", files)
                    # print(f"Folder Name: {final_folder_name}")

                    # Open the file and read its contents
                    dataset = h5py.File(file_path, 'r')

                    # for verification purpose only
                    processed_data = dataset['Processed']
                    processed_data.keys()

                    sensorslist = dataset['Sensors']
                    sensorslist.keys()

                    for i in sensorslist.keys():
                        # different sensors
                        if (i == '7279'): # skip left upper arm sensor data as it is not used
                            continue

                        sensor = sensorslist[i]
                        sensor_orien = processed_data[i]

                        # add other variables for extraction
                        accelerometer_values = np.array(list(sensor['Accelerometer']))
                        accelerometer_values_x = accelerometer_values[:,0]
                        accelerometer_values_y = accelerometer_values[:,1]
                        accelerometer_values_z = accelerometer_values[:,2]
                        orientation_values = np.array(list(sensor_orien['Orientation']))
                        orientation_coefficient = orientation_values[:,0]
                        orientation_i = orientation_values[:,1]
                        orientation_j = orientation_values[:,2]
                        orientation_k = orientation_values[:,3]
                        gyroscope_values = np.array(list(sensor['Gyroscope']))
                        gyroscope_values_x = gyroscope_values[:,0]
                        gyroscope_values_y = gyroscope_values[:,1]
                        gyroscope_values_z = gyroscope_values[:,2]

                        h5_dataframe = pd.DataFrame({"Acceleration x-axis": accelerometer_values_x, "Acceleration y-axis": accelerometer_values_y, "acceleration z-axis": accelerometer_values_z,
                                                "Angular Velocity x-axis": gyroscope_values_x, "Angular Velocity y-axis": gyroscope_values_y, "Angular Velocity z-axis": gyroscope_values_z,
                                                "Quaternion Coefficient": orientation_coefficient, "Quaternion i-axis": orientation_i, "Quaternion j-axis": orientation_j, "Quaternion k-axis": orientation_k})
                        h5_all_dataframe = pd.concat([h5_all_dataframe, h5_dataframe], ignore_index = True, axis = 1)

                    file_number += 1

                elif (file_name.endswith('_Joint_Angles.csv')):

                    file_name_without_extension = os.path.splitext(file_name)[0]

                    # print("File path:", file_path)
                    # print("Root:", root)
                    print(f'File {file_number}: {file_name}')
                    # print("Dir:", dirs)
                    # print("File name without extension:", file_name_without_extension)
                    # print("Files:", files)
                    # print(f"Folder Name: {final_folder_name}")
                    
                    df = pd.read_csv(file_path, header = 14)
                    joint_angle_array = np.array(df.values.tolist())

                    # elbow_sup_pro_left = joint_angle_array[:,14]
                    # elbow_flx_ext_left = joint_angle_array[:,15]
                    elbow_sup_pro_right = joint_angle_array[:,16]
                    elbow_flx_ext_right = joint_angle_array[:,17]
                    # wrist_flx_ext_left = joint_angle_array[:,18]
                    # wrist_uln_rad_left = joint_angle_array[:,19]
                    wrist_flx_ext_right = joint_angle_array[:,20]
                    wrist_uln_rad_right = joint_angle_array[:,21]

                    csv_dataframe = pd.DataFrame({#"Elbow (L) Pronation & Supination": elbow_sup_pro_left, "Elbow (L) Flexion & Extension": elbow_flx_ext_left,
                                                    "Elbow - Pronation & Supination": elbow_sup_pro_right, "Elbow - Flexion & Extension": elbow_flx_ext_right,
                                                    #"Wrist (L) Flexion & Extension": wrist_flx_ext_left, "Wrist (L) Ulnar & Radial Deviation": wrist_uln_rad_left,
                                                    "Wrist - Flexion & Extension": wrist_flx_ext_right, "Wrist - Ulnar & Radial Deviation": wrist_uln_rad_right})
                    csv_all_dataframe = pd.concat([csv_all_dataframe, csv_dataframe], ignore_index = True, axis = 1)
                
                    file_number += 1

        print(f'Total number of files read: {file_number-1}')
        print(f'Size of combined h5 dataframes (motion data): {np.shape(h5_all_dataframe)}')
        print(f'Size of combined csv dataframes (joint angles): {np.shape(csv_all_dataframe)}')

        # Naming order: 10464 (I), 10468 (U), 10833 (T), 10871 (L), 7257 (H)
        h5_all_dataframe.columns = pd.MultiIndex.from_tuples([
            ('-', '-', 'Device Time (milliseconds)'), 
            ('-', '-', 'Time'),

            ('REST (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('REST (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('REST (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('REST (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('REST (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('REST (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('REST (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('REST (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('REST (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('REST (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('REST (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('REST (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('REST (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('REST (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('REST (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('REST (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('REST (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('REST (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('REST (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('REST (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('REST (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('REST (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('REST (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('REST (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('REST (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('REST (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('REST (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('REST (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('REST (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('REST (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('REST (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('REST (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('REST (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('REST (R)', 'Not Used (7279)', 'Quaternion k-axis'), 
            
            ('OUT (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('OUT (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('OUT (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('OUT (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('OUT (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('OUT (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('OUT (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('OUT (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('OUT (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('OUT (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('OUT (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('OUT (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('OUT (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('OUT (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('OUT (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('OUT (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('OUT (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('OUT (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('OUT (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('OUT (R)', 'Not Used (7279)', 'Quaternion k-axis'),

            ('DRINK (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('DRINK (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('DRINK (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('DRINK (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('DRINK (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('DRINK (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('DRINK (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('DRINK (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('DRINK (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion k-axis'),

            ('FNF (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('FNF (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('FNF (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('FNF (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('FNF (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FNF (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FNF (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FNF (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FNF (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FNF (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('FNF (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('FNF (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('FNF (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('FNF (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FNF (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FNF (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FNF (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FNF (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('FNF (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('FNF (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('POUR (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('POUR (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('POUR (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('POUR (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('POUR (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('POUR (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('POUR (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('POUR (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('POUR (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('POUR (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('POUR (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('POUR (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('POUR (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('POUR (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('POUR (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('POUR (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('POUR (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('POUR (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('POUR (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('POUR (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('TAP (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('TAP (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('TAP (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('TAP (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('TAP (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('TAP (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('TAP (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('TAP (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('TAP (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('TAP (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('TAP (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('TAP (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('TAP (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('TAP (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('TAP (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('TAP (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('TAP (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('TAP (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('TAP (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('TAP (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('FLIP (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('FLIP (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('FLIP (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('FLIP (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('FLIP (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('FLIP (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('FLIP (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('FLIP (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('FLIP (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('REST (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('REST (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('REST (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('REST (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('REST (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('REST (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('REST (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('REST (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('REST (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('REST (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('REST (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('REST (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('REST (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('REST (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('REST (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('REST (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('REST (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('REST (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('REST (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('REST (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('REST (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('REST (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('REST (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('REST (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('REST (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('REST (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('REST (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('REST (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('REST (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('REST (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('REST (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('REST (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('REST (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('REST (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('OUT (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('OUT (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('OUT (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('OUT (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('OUT (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('OUT (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('OUT (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('OUT (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('OUT (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('OUT (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('OUT (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('OUT (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('OUT (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('OUT (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('OUT (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('OUT (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('OUT (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('OUT (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('OUT (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('OUT (L)', 'Not Used (7279)', 'Quaternion k-axis'),

            ('DRINK (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('DRINK (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('DRINK (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('DRINK (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('DRINK (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('DRINK (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('DRINK (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('DRINK (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('DRINK (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion k-axis'),

            ('FNF (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('FNF (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('FNF (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('FNF (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('FNF (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FNF (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FNF (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FNF (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FNF (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FNF (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('FNF (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('FNF (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('FNF (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('FNF (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FNF (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FNF (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FNF (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FNF (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('FNF (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('FNF (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('POUR (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('POUR (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('POUR (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('POUR (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('POUR (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('POUR (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('POUR (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('POUR (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('POUR (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('POUR (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('POUR (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('POUR (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('POUR (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('POUR (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('POUR (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('POUR (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('POUR (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('POUR (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('POUR (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('POUR (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('TAP (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('TAP (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('TAP (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('TAP (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('TAP (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('TAP (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('TAP (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('TAP (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('TAP (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('TAP (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('TAP (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('TAP (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('TAP (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('TAP (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('TAP (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('TAP (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('TAP (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('TAP (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('TAP (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('TAP (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ('FLIP (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
            ('FLIP (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
            ('FLIP (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
            ('FLIP (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
            ('FLIP (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
            # ('FLIP (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('FLIP (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('FLIP (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('FLIP (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

            ], names = ['Movement', 'Sensor', 'Data'])

        csv_all_dataframe.columns = pd.MultiIndex.from_tuples([
            ('-', 'Device Time (milliseconds)'), 
            ('-', 'Time'),
            
            ('REST (R)', 'Elbow - Pronation & Supination'), ('REST (R)', 'Elbow - Flexion & Extension'), ('REST (R)', 'Wrist - Flexion & Extension'), ('REST (R)', 'Wrist - Ulnar & Radial Deviation'), 
            ('OUT (R)', 'Elbow - Pronation & Supination'), ('OUT (R)', 'Elbow - Flexion & Extension'), ('OUT (R)', 'Wrist - Flexion & Extension'), ('OUT (R)', 'Wrist - Ulnar & Radial Deviation'), 
            ('DRINK (R)', 'Elbow - Pronation & Supination'), ('DRINK (R)', 'Elbow - Flexion & Extension'), ('DRINK (R)', 'Wrist - Flexion & Extension'), ('DRINK (R)', 'Wrist - Ulnar & Radial Deviation'), 
            ('FNF (R)', 'Elbow - Pronation & Supination'), ('FNF (R)', 'Elbow - Flexion & Extension'), ('FNF (R)', 'Wrist - Flexion & Extension'), ('FNF (R)', 'Wrist - Ulnar & Radial Deviation'), 
            ('POUR (R)', 'Elbow - Pronation & Supination'), ('POUR (R)', 'Elbow - Flexion & Extension'), ('POUR (R)', 'Wrist - Flexion & Extension'), ('POUR (R)', 'Wrist - Ulnar & Radial Deviation'), 
            ('TAP (R)', 'Elbow - Pronation & Supination'), ('TAP (R)', 'Elbow - Flexion & Extension'), ('TAP (R)', 'Wrist - Flexion & Extension'), ('TAP (R)', 'Wrist - Ulnar & Radial Deviation'), 
            ('FLIP (R)', 'Elbow - Pronation & Supination'), ('FLIP (R)', 'Elbow - Flexion & Extension'), ('FLIP (R)', 'Wrist - Flexion & Extension'), ('FLIP (R)', 'Wrist - Ulnar & Radial Deviation'), 

            ('REST (L)', 'Elbow - Pronation & Supination'), ('REST (L)', 'Elbow - Flexion & Extension'), ('REST (L)', 'Wrist - Flexion & Extension'), ('REST (L)', 'Wrist - Ulnar & Radial Deviation'), 
            ('OUT (L)', 'Elbow - Pronation & Supination'), ('OUT (L)', 'Elbow - Flexion & Extension'), ('OUT (L)', 'Wrist - Flexion & Extension'), ('OUT (L)', 'Wrist - Ulnar & Radial Deviation'), 
            ('DRINK (L)', 'Elbow - Pronation & Supination'), ('DRINK (L)', 'Elbow - Flexion & Extension'), ('DRINK (L)', 'Wrist - Flexion & Extension'), ('DRINK (L)', 'Wrist - Ulnar & Radial Deviation'), 
            ('FNF (L)', 'Elbow - Pronation & Supination'), ('FNF (L)', 'Elbow - Flexion & Extension'), ('FNF (L)', 'Wrist - Flexion & Extension'), ('FNF (L)', 'Wrist - Ulnar & Radial Deviation'), 
            ('POUR (L)', 'Elbow - Pronation & Supination'), ('POUR (L)', 'Elbow - Flexion & Extension'), ('POUR (L)', 'Wrist - Flexion & Extension'), ('POUR (L)', 'Wrist - Ulnar & Radial Deviation'), 
            ('TAP (L)', 'Elbow - Pronation & Supination'), ('TAP (L)', 'Elbow - Flexion & Extension'), ('TAP (L)', 'Wrist - Flexion & Extension'), ('TAP (L)', 'Wrist - Ulnar & Radial Deviation'), 
            ('FLIP (L)', 'Elbow - Pronation & Supination'), ('FLIP (L)', 'Elbow - Flexion & Extension'), ('FLIP (L)', 'Wrist - Flexion & Extension'), ('FLIP (L)', 'Wrist - Ulnar & Radial Deviation'), 

        ], names = ['Movement', 'Joint Angle'])

        DCS_file_directory = 'C:/Users/adria/OneDrive - Monash University/1 Raw Data/Clinical Study Subjects - Non-Identifiable Data.csv'
        df_DCS = pd.read_csv(DCS_file_directory, header = 2)
        DCS_array = np.array(df_DCS.values.tolist())
        DCS_subject_codes = DCS_array[:, 1]
        subject_index = np.where(np.char.find(DCS_subject_codes, subject_code) >= 0) # np.where(DCS_subject_codes==subject_code)
        if subject_index[0].size == 0:
            print(f'{subject_code} cannot be found from the DCS, therefore, the {subject_code} excel file cannot be created!!!\n')
            continue

        subject_study_site = DCS_array[subject_index[0][0], 2]
        subject_age = DCS_array[subject_index[0][0], 3]
        subject_gender = DCS_array[subject_index[0][0], 4]
        subject_race = DCS_array[subject_index[0][0], 5]
        subject_height = DCS_array[subject_index[0][0], 6]

        subject_weight = DCS_array[subject_index[0][0], 7]
        subject_dominant_hand = DCS_array[subject_index[0][0], 8]
        subject_h_y_before = DCS_array[subject_index[0][0], 9]
        subject_h_y_after = DCS_array[subject_index[0][0], 10]
        subject_remark_incl_excl = DCS_array[subject_index[0][0], 11]

        subject_remark_procedure = DCS_array[subject_index[0][0], 12]
        subject_rigidity = DCS_array[subject_index[0][0], 13]
        subject_remark_ET_diagnosis = DCS_array[subject_index[0][0], 14]
        subject_motor_symptoms = DCS_array[subject_index[0][0], 15]
        subject_alcohol_on_tremor = DCS_array[subject_index[0][0], 16]

        subject_fam_hist = DCS_array[subject_index[0][0], 17]
        subject_onset_symptoms = DCS_array[subject_index[0][0], 18]
        subject_duration_onset = DCS_array[subject_index[0][0], 19]
        subject_diag_year = DCS_array[subject_index[0][0], 20]
        subject_diag_duration = DCS_array[subject_index[0][0], 21]

        subject_tremor_more = DCS_array[subject_index[0][0], 22]
        subject_brady_more = DCS_array[subject_index[0][0], 23]
        subject_last_dose = DCS_array[subject_index[0][0], 24]
        subject_participation_time = DCS_array[subject_index[0][0], 25]
        subject_duration_no_med = DCS_array[subject_index[0][0], 26]

        subject_prescribed_med = DCS_array[subject_index[0][0], 27]
        subject_intake_freq = DCS_array[subject_index[0][0], 28]
        subject_left_upper = DCS_array[subject_index[0][0], 29]
        subject_left_fore = DCS_array[subject_index[0][0], 30]
        subject_left_hand = DCS_array[subject_index[0][0], 31]

        subject_right_upper = DCS_array[subject_index[0][0], 32]
        subject_right_fore = DCS_array[subject_index[0][0], 33]
        subject_right_hand = DCS_array[subject_index[0][0], 34]

        subject_info_dataframe = pd.DataFrame(columns = ['', ''])

        subject_info_dataframe.loc[0] = ["Subject Group", subject_group]
        subject_info_dataframe.loc[1] = ["Subject Code", subject_code]
        subject_info_dataframe.loc[2] = ["Study Site", subject_study_site]
        subject_info_dataframe.loc[3] = ["Age", subject_age]
        subject_info_dataframe.loc[4] = ["Gender", subject_gender]

        subject_info_dataframe.loc[5] = ["Race", subject_race]
        subject_info_dataframe.loc[6] = ["Height (cm)", subject_height]
        subject_info_dataframe.loc[7] = ["Weight (kg)", subject_weight]
        subject_info_dataframe.loc[8] = ["Dominant Hand", subject_dominant_hand]
        subject_info_dataframe.loc[9] = ["H&Y stage (before)", subject_h_y_before]

        subject_info_dataframe.loc[10] = ["H&Y stage (after)", subject_h_y_after]         
        subject_info_dataframe.loc[11] = ["Remark on Inclusion & Exclusion Criteria", subject_remark_incl_excl]
        subject_info_dataframe.loc[12] = ["Remark on Clinical Study Procedure", subject_remark_procedure]
        subject_info_dataframe.loc[13] = ["Rigidity Test", subject_rigidity]
        subject_info_dataframe.loc[14] = ["Remark on Diagnosis of ET", subject_remark_ET_diagnosis]

        subject_info_dataframe.loc[15] = ["Motor Symptoms", subject_motor_symptoms]        
        subject_info_dataframe.loc[16] = ["Tremor's Response on Intake of Alcohol", subject_alcohol_on_tremor]
        subject_info_dataframe.loc[17] = ["Family History", subject_fam_hist]
        subject_info_dataframe.loc[18] = ["Onset of Motor Symptoms", subject_onset_symptoms]
        subject_info_dataframe.loc[19] = ["Duration Since Onset of Motor Symptoms", subject_duration_onset]

        subject_info_dataframe.loc[20] = ["Year of Diagnosis", subject_diag_year]
        subject_info_dataframe.loc[21] = ["Duration Since Diagnosis", subject_diag_duration]
        subject_info_dataframe.loc[22] = ["Tremor More Affected Hand", subject_tremor_more]
        subject_info_dataframe.loc[23] = ["Bradykinesia More Affected Hand", subject_brady_more]
        subject_info_dataframe.loc[24] = ["Last Dose", subject_last_dose]

        subject_info_dataframe.loc[25] = ["Study Start Time", subject_participation_time]
        subject_info_dataframe.loc[26] = ["Duration between Last Dose and Study Start Time", subject_duration_no_med]
        subject_info_dataframe.loc[27] = ["Prescribed Medications", subject_prescribed_med]
        subject_info_dataframe.loc[28] = ["Medications Intake Frequency", subject_intake_freq]
        subject_info_dataframe.loc[29] = ["Length of Upper Arm (L)", subject_left_upper]
        
        subject_info_dataframe.loc[30] = ["Length of Forearm (L)", subject_left_fore]
        subject_info_dataframe.loc[31] = ["Length of Hand (L)", subject_left_hand]
        subject_info_dataframe.loc[32] = ["Length of Upper Arm (R)", subject_right_upper]
        subject_info_dataframe.loc[33] = ["Length of Forearm (R)", subject_right_fore]
        subject_info_dataframe.loc[34] = ["Length of Hand (R)", subject_right_hand]

        print(f'Size of subject info dataframes (DCS): {np.shape(subject_info_dataframe)}')

        with pd.ExcelWriter(new_excel_file_path) as writer:
            subject_info_dataframe.to_excel(writer, sheet_name='Data Collection Sheet', header = True)
            h5_all_dataframe.to_excel(writer, sheet_name='Motion Data', header = True)
            csv_all_dataframe.to_excel(writer, sheet_name='Joint Angles', header = True)
                
        end_time = time.time()
        elapsed_time = end_time - start_time
        if (elapsed_time >= 60):
            elapsed_minute = int(np.floor(elapsed_time/60))
            elapsed_second = int(np.ceil(elapsed_time - elapsed_minute*60))
            if (elapsed_minute > 1):
                print(f'{subject_code} excel file has been successfully created, which took {elapsed_minute} minutes {elapsed_second} seconds.\n')
            else:
                print(f'{subject_code} excel file has been successfully created, which took {elapsed_minute} minute {elapsed_second} seconds.\n')
        else:
            elapsed_time = int(np.ceil(elapsed_time))
            print(f'{subject_code} excel file has been successfully created, which took {elapsed_time} seconds.\n')

print(f"All subjects' excel file have been successfully created!\n")


Subject Code: ET 19
ET 19.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 68
HC 68.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 69
HC 69.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 70
HC 70.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 71
HC 71.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 72
HC 72.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 73
HC 73.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 74
HC 74.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 75
HC 75.xlsx has already been created, proceeding with the next subject.

Subject Code: HC 76
HC 76.xlsx has already been created, proceeding with the next subject.

Subject Code: PD 100
PD 100.xlsx has already been created, proceeding with the n

In [5]:
# run until the cell above

Converting h5 and csv to xlsx for one subject

In [ ]:
# Define the root directory where your files are located
root_directory = 'C:/Users/adria/OneDrive - Monash University/1 Raw Data/Moveo_Explorer_Subject_Export_HC_68_20241226-160610SGT'

# Define the specific file types you want to read
file_types = ['.h5', '_Joint_Angles.csv']  # Add more file extensions as needed

csv_all_dataframe, h5_all_dataframe, time_dataframe = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# To find the h5 file with the longest recordings
time_array_length = 0
longest_device_time_array = []
longest_time_array = []

for root, dirs, files in os.walk(root_directory):
  for file_name in files:
    
    start_time = time.time()

    # Construct the absolute path of the file
    file_path = os.path.join(root, file_name)

    # Check if the file's extension matches any of the specified file types
    if (file_name.endswith('.h5') and re.search(r'\d+\.h5$', file_name)):

      # Open the file and read its contents
      dataset = h5py.File(file_path, 'r')

      sensorslist = dataset['Sensors']
      sensor_list_keys = list(sensorslist.keys())

      sensor = sensorslist[sensor_list_keys[0]] # take any sensor will do since same recording all 6 sensors will have same length of recording

      time_values = np.array(list(sensor['Time']))
      #converted_time = array.array('f')
      converted_time = np.array([None] * len(time_values))
      for j in range(len(time_values)):
        if j == 0:
          time_zero = time_values[j]
          converted_time[j] = 0
        else:
          converted_time[j] = (time_values[j] - time_zero)/(1000*1000)

      if (len(time_values) > time_array_length):
        longest_device_time_array = time_values
        longest_time_array = converted_time
        time_array_length = len(time_values)

time_dataframe = pd.DataFrame({"Device Time (milliseconds)": longest_device_time_array, "Time": longest_time_array})
h5_all_dataframe = pd.concat([h5_all_dataframe, time_dataframe], ignore_index = True, axis = 1)
csv_all_dataframe = pd.concat([csv_all_dataframe, time_dataframe], ignore_index = True, axis = 1)

# find the subject code from the file path
subject_code_match = re.search(r'(HC_\d+|PD_\d+|ET_\d+)', root_directory)
if subject_code_match:
  subject_code = subject_code_match.group(0)  # Extract the matched pattern
  subject_code = subject_code.replace("_", " ")
  print(f"Subject Code: {subject_code}")
else:
  print("No valid subject code found in the file path.")
    
# find the subject group from the file path
subject_group_match = re.search(r'(_HC_+|_PD_+|_ET_+)', root_directory)
if subject_group_match:
  subject_group = subject_group_match.group(0)  # Extract the matched pattern
  subject_group = subject_group.replace("_", "")
else:
  print("No valid subject group found in the file path.")

file_number = 1
# Walk through all the directories and files starting from the root directory
for root, dirs, files in os.walk(root_directory):
  for file_name in files:
    # Construct the absolute path of the file
    file_path = os.path.join(root, file_name)

    # Check if the file's extension matches any of the specified file types
    if (file_name.endswith('.h5') and re.search(r'\d+\.h5$', file_name)):

      file_name_without_extension = os.path.splitext(file_name)[0]

      # Get the final folder name in the root path
      #final_folder_name = os.path.basename(root)

      # print("File path:", file_path)
      # print("Root:", root)
      print(f'File name {file_number}: {file_name}')
      # print("Dir:", dirs)
      # print("File name without extension:", file_name_without_extension)
      # print("Files:", files)
      # print(f"Folder Name: {final_folder_name}")

      # Open the file and read its contents
      dataset = h5py.File(file_path, 'r')

      # for verification purpose only
      processed_data = dataset['Processed']
      processed_data.keys()

      sensorslist = dataset['Sensors']
      sensorslist.keys()

      for i in sensorslist.keys():

        if (i == '7279'): # skip left upper arm sensor data as it is not used
          continue

        # different sensors
        sensor = sensorslist[i]
        sensor_orien = processed_data[i]

        # add other variables for extraction
        accelerometer_values = np.array(list(sensor['Accelerometer']))
        accelerometer_values_x = accelerometer_values[:,0]
        accelerometer_values_y = accelerometer_values[:,1]
        accelerometer_values_z = accelerometer_values[:,2]
        orientation_values = np.array(list(sensor_orien['Orientation']))
        orientation_coefficient = orientation_values[:,0]
        orientation_i = orientation_values[:,1]
        orientation_j = orientation_values[:,2]
        orientation_k = orientation_values[:,3]
        gyroscope_values = np.array(list(sensor['Gyroscope']))
        gyroscope_values_x = gyroscope_values[:,0]
        gyroscope_values_y = gyroscope_values[:,1]
        gyroscope_values_z = gyroscope_values[:,2]

        h5_dataframe = pd.DataFrame({"Acceleration x-axis": accelerometer_values_x, "Acceleration y-axis": accelerometer_values_y, "acceleration z-axis": accelerometer_values_z,
                                  "Angular Velocity x-axis": gyroscope_values_x, "Angular Velocity y-axis": gyroscope_values_y, "Angular Velocity z-axis": gyroscope_values_z,
                                  "Quaternion Coefficient": orientation_coefficient, "Quaternion i-axis": orientation_i, "Quaternion j-axis": orientation_j, "Quaternion k-axis": orientation_k})
        h5_all_dataframe = pd.concat([h5_all_dataframe, h5_dataframe], ignore_index = True, axis = 1)

      file_number += 1

    elif (file_name.endswith('_Joint_Angles.csv')):

      file_name_without_extension = os.path.splitext(file_name)[0]

      # print("File path:", file_path)
      # print("Root:", root)
      print(f'File name {file_number}: {file_name}')
      # print("Dir:", dirs)
      # print("File name without extension:", file_name_without_extension)
      # print("Files:", files)
      # print(f"Folder Name: {final_folder_name}")
      
      df = pd.read_csv(file_path, header = 14)
      joint_angle_array = np.array(df.values.tolist())

      # elbow_sup_pro_left = joint_angle_array[:,14]
      # elbow_flx_ext_left = joint_angle_array[:,15]
      elbow_sup_pro_right = joint_angle_array[:,16]
      elbow_flx_ext_right = joint_angle_array[:,17]
      # wrist_flx_ext_left = joint_angle_array[:,18]
      # wrist_uln_rad_left = joint_angle_array[:,19]
      wrist_flx_ext_right = joint_angle_array[:,20]
      wrist_uln_rad_right = joint_angle_array[:,21]

      csv_dataframe = pd.DataFrame({#"Elbow (L) Pronation & Supination": elbow_sup_pro_left, "Elbow (L) Flexion & Extension": elbow_flx_ext_left,
                                    "Elbow - Pronation & Supination": elbow_sup_pro_right, "Elbow - Flexion & Extension": elbow_flx_ext_right,
                                    #"Wrist (L) Flexion & Extension": wrist_flx_ext_left, "Wrist (L) Ulnar & Radial Deviation": wrist_uln_rad_left,
                                    "Wrist - Flexion & Extension": wrist_flx_ext_right, "Wrist - Ulnar & Radial Deviation": wrist_uln_rad_right})
      csv_all_dataframe = pd.concat([csv_all_dataframe, csv_dataframe], ignore_index = True, axis = 1)
  
      file_number += 1

print(f'Total number of files: {file_number - 1}')
print(f'Size of combined h5 dataframes: {np.shape(h5_all_dataframe)}')
print(f'Size of combined csv dataframes: {np.shape(csv_all_dataframe)}')

# Naming order: 10464 (I), 10468 (U), 10833 (T), 10871 (L), 7257 (H)
h5_all_dataframe.columns = pd.MultiIndex.from_tuples([
    ('-', '-', 'Device Time (milliseconds)'), 
    ('-', '-', 'Time'),

    ('REST (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('REST (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('REST (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('REST (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('REST (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('REST (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('REST (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('REST (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('REST (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('REST (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('REST (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('REST (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('REST (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('REST (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('REST (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('REST (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('REST (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('REST (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('REST (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('REST (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('REST (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('REST (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('REST (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('REST (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('REST (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('REST (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('REST (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('REST (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('REST (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('REST (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('REST (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('REST (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('REST (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('REST (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('REST (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('REST (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('REST (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('REST (R)', 'Not Used (7279)', 'Quaternion k-axis'), 
    
    ('OUT (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('OUT (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('OUT (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('OUT (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('OUT (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('OUT (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('OUT (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('OUT (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('OUT (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('OUT (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('OUT (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('OUT (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('OUT (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('OUT (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('OUT (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('OUT (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('OUT (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('OUT (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('OUT (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('OUT (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('OUT (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('OUT (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('OUT (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('OUT (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('OUT (R)', 'Not Used (7279)', 'Quaternion k-axis'),

    ('DRINK (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('DRINK (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('DRINK (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('DRINK (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('DRINK (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('DRINK (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('DRINK (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('DRINK (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('DRINK (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('DRINK (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('DRINK (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('DRINK (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('DRINK (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('DRINK (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('DRINK (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('DRINK (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('DRINK (R)', 'Not Used (7279)', 'Quaternion k-axis'),

    ('FNF (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FNF (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('FNF (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FNF (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('FNF (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FNF (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('FNF (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FNF (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('FNF (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FNF (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FNF (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FNF (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FNF (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FNF (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FNF (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('FNF (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('FNF (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('FNF (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('FNF (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FNF (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FNF (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FNF (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FNF (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('FNF (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('FNF (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('POUR (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('POUR (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('POUR (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('POUR (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('POUR (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('POUR (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('POUR (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('POUR (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('POUR (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('POUR (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('POUR (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('POUR (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('POUR (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('POUR (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('POUR (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('POUR (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('POUR (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('POUR (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('POUR (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('POUR (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('POUR (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('POUR (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('POUR (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('POUR (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('POUR (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('TAP (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('TAP (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('TAP (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('TAP (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('TAP (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('TAP (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('TAP (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('TAP (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('TAP (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('TAP (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('TAP (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('TAP (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('TAP (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('TAP (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('TAP (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('TAP (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('TAP (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('TAP (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('TAP (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('TAP (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('TAP (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('TAP (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('TAP (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('TAP (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('TAP (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('FLIP (R)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FLIP (R)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('FLIP (R)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FLIP (R)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('FLIP (R)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FLIP (R)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('FLIP (R)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FLIP (R)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('FLIP (R)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FLIP (R)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('FLIP (R)', 'Not Used (7279)', 'Acceleration x-axis'), ('FLIP (R)', 'Not Used (7279)', 'Acceleration y-axis'), ('FLIP (R)', 'Not Used (7279)', 'Acceleration z-axis'), ('FLIP (R)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FLIP (R)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FLIP (R)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion i-axis'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion j-axis'), ('FLIP (R)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('REST (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('REST (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('REST (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('REST (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('REST (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('REST (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('REST (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('REST (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('REST (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('REST (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('REST (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('REST (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('REST (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('REST (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('REST (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('REST (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('REST (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('REST (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('REST (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('REST (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('REST (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('REST (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('REST (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('REST (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('REST (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('REST (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('REST (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('REST (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('REST (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('REST (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('REST (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('REST (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('REST (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('REST (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('REST (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('REST (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('REST (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('REST (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('OUT (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('OUT (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('OUT (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('OUT (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('OUT (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('OUT (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('OUT (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('OUT (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('OUT (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('OUT (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('OUT (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('OUT (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('OUT (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('OUT (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('OUT (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('OUT (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('OUT (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('OUT (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('OUT (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('OUT (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('OUT (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('OUT (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('OUT (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('OUT (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('OUT (L)', 'Not Used (7279)', 'Quaternion k-axis'),

    ('DRINK (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('DRINK (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('DRINK (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('DRINK (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('DRINK (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('DRINK (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('DRINK (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('DRINK (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('DRINK (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('DRINK (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('DRINK (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('DRINK (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('DRINK (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('DRINK (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('DRINK (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('DRINK (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('DRINK (L)', 'Not Used (7279)', 'Quaternion k-axis'),

    ('FNF (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FNF (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('FNF (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FNF (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('FNF (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FNF (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('FNF (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FNF (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('FNF (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FNF (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FNF (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FNF (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FNF (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FNF (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FNF (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('FNF (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('FNF (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('FNF (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('FNF (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FNF (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FNF (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FNF (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FNF (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('FNF (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('FNF (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('POUR (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('POUR (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('POUR (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('POUR (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('POUR (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('POUR (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('POUR (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('POUR (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('POUR (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('POUR (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('POUR (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('POUR (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('POUR (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('POUR (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('POUR (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('POUR (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('POUR (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('POUR (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('POUR (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('POUR (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('POUR (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('POUR (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('POUR (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('POUR (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('POUR (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('TAP (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('TAP (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('TAP (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('TAP (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('TAP (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('TAP (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('TAP (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('TAP (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('TAP (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('TAP (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('TAP (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('TAP (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('TAP (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('TAP (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('TAP (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('TAP (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('TAP (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('TAP (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('TAP (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('TAP (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('TAP (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('TAP (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('TAP (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('TAP (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('TAP (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ('FLIP (L)', 'Index Finger - I (10464)', 'Acceleration x-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Acceleration y-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Acceleration z-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion Coefficient'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion i-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion j-axis'), ('FLIP (L)', 'Index Finger - I (10464)', 'Quaternion k-axis'),
    ('FLIP (L)', 'Upper Arm - U (10468)', 'Acceleration x-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Acceleration y-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Acceleration z-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion Coefficient'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion i-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion j-axis'), ('FLIP (L)', 'Upper Arm - U (10468)', 'Quaternion k-axis'), 
    ('FLIP (L)', 'Thumb - T (10833)', 'Acceleration x-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Acceleration y-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Acceleration z-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion Coefficient'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion i-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion j-axis'), ('FLIP (L)', 'Thumb - T (10833)', 'Quaternion k-axis'), 
    ('FLIP (L)', 'Wrist - L (10871)', 'Acceleration x-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Acceleration y-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Acceleration z-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion Coefficient'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion i-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion j-axis'), ('FLIP (L)', 'Wrist - L (10871)', 'Quaternion k-axis'), 
    ('FLIP (L)', 'Hand - H (7257)', 'Acceleration x-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Acceleration y-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Acceleration z-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion Coefficient'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion i-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion j-axis'), ('FLIP (L)', 'Hand - H (7257)', 'Quaternion k-axis'),  
    # ('FLIP (L)', 'Not Used (7279)', 'Acceleration x-axis'), ('FLIP (L)', 'Not Used (7279)', 'Acceleration y-axis'), ('FLIP (L)', 'Not Used (7279)', 'Acceleration z-axis'), ('FLIP (L)', 'Not Used (7279)', 'Angular Velocity x-axis'), ('FLIP (L)', 'Not Used (7279)', 'Angular Velocity y-axis'), ('FLIP (L)', 'Not Used (7279)', 'Angular Velocity z-axis'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion Coefficient'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion i-axis'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion j-axis'), ('FLIP (L)', 'Not Used (7279)', 'Quaternion k-axis'), 

    ], names = ['Movement', 'Sensor', 'Data'])

csv_all_dataframe.columns = pd.MultiIndex.from_tuples([
    ('-', 'Device Time (milliseconds)'), 
    ('-', 'Time'),
    
    ('REST (R)', 'Elbow - Pronation & Supination'), ('REST (R)', 'Elbow - Flexion & Extension'), ('REST (R)', 'Wrist - Flexion & Extension'), ('REST (R)', 'Wrist - Ulnar & Radial Deviation'), 
    ('OUT (R)', 'Elbow - Pronation & Supination'), ('OUT (R)', 'Elbow - Flexion & Extension'), ('OUT (R)', 'Wrist - Flexion & Extension'), ('OUT (R)', 'Wrist - Ulnar & Radial Deviation'), 
    ('DRINK (R)', 'Elbow - Pronation & Supination'), ('DRINK (R)', 'Elbow - Flexion & Extension'), ('DRINK (R)', 'Wrist - Flexion & Extension'), ('DRINK (R)', 'Wrist - Ulnar & Radial Deviation'), 
    ('FNF (R)', 'Elbow - Pronation & Supination'), ('FNF (R)', 'Elbow - Flexion & Extension'), ('FNF (R)', 'Wrist - Flexion & Extension'), ('FNF (R)', 'Wrist - Ulnar & Radial Deviation'), 
    ('POUR (R)', 'Elbow - Pronation & Supination'), ('POUR (R)', 'Elbow - Flexion & Extension'), ('POUR (R)', 'Wrist - Flexion & Extension'), ('POUR (R)', 'Wrist - Ulnar & Radial Deviation'), 
    ('TAP (R)', 'Elbow - Pronation & Supination'), ('TAP (R)', 'Elbow - Flexion & Extension'), ('TAP (R)', 'Wrist - Flexion & Extension'), ('TAP (R)', 'Wrist - Ulnar & Radial Deviation'), 
    ('FLIP (R)', 'Elbow - Pronation & Supination'), ('FLIP (R)', 'Elbow - Flexion & Extension'), ('FLIP (R)', 'Wrist - Flexion & Extension'), ('FLIP (R)', 'Wrist - Ulnar & Radial Deviation'), 

    ('REST (L)', 'Elbow - Pronation & Supination'), ('REST (L)', 'Elbow - Flexion & Extension'), ('REST (L)', 'Wrist - Flexion & Extension'), ('REST (L)', 'Wrist - Ulnar & Radial Deviation'), 
    ('OUT (L)', 'Elbow - Pronation & Supination'), ('OUT (L)', 'Elbow - Flexion & Extension'), ('OUT (L)', 'Wrist - Flexion & Extension'), ('OUT (L)', 'Wrist - Ulnar & Radial Deviation'), 
    ('DRINK (L)', 'Elbow - Pronation & Supination'), ('DRINK (L)', 'Elbow - Flexion & Extension'), ('DRINK (L)', 'Wrist - Flexion & Extension'), ('DRINK (L)', 'Wrist - Ulnar & Radial Deviation'), 
    ('FNF (L)', 'Elbow - Pronation & Supination'), ('FNF (L)', 'Elbow - Flexion & Extension'), ('FNF (L)', 'Wrist - Flexion & Extension'), ('FNF (L)', 'Wrist - Ulnar & Radial Deviation'), 
    ('POUR (L)', 'Elbow - Pronation & Supination'), ('POUR (L)', 'Elbow - Flexion & Extension'), ('POUR (L)', 'Wrist - Flexion & Extension'), ('POUR (L)', 'Wrist - Ulnar & Radial Deviation'), 
    ('TAP (L)', 'Elbow - Pronation & Supination'), ('TAP (L)', 'Elbow - Flexion & Extension'), ('TAP (L)', 'Wrist - Flexion & Extension'), ('TAP (L)', 'Wrist - Ulnar & Radial Deviation'), 
    ('FLIP (L)', 'Elbow - Pronation & Supination'), ('FLIP (L)', 'Elbow - Flexion & Extension'), ('FLIP (L)', 'Wrist - Flexion & Extension'), ('FLIP (L)', 'Wrist - Ulnar & Radial Deviation'), 

], names = ['Movement', 'Joint Angle'])

DCS_file_directory = 'C:/Users/adria/OneDrive - Monash University/1 Raw Data/Clinical Study Subjects - Non-Identifiable Data.csv'
df_DCS = pd.read_csv(DCS_file_directory, header = 2)
DCS_array = np.array(df_DCS.values.tolist())
DCS_subject_codes = DCS_array[:, 1]
subject_index = np.where(np.char.find(DCS_subject_codes, subject_code) >= 0) # np.where(DCS_subject_codes==subject_code)
if subject_index[0].size == 0:
  print(f'{subject_code} cannot be found from the DCS, therefore, the {subject_code} excel file cannot be created!!!\n')

subject_study_site = DCS_array[subject_index[0][0], 2]
subject_age = DCS_array[subject_index[0][0], 3]
subject_gender = DCS_array[subject_index[0][0], 4]
subject_race = DCS_array[subject_index[0][0], 5]
subject_height = DCS_array[subject_index[0][0], 6]

subject_weight = DCS_array[subject_index[0][0], 7]
subject_dominant_hand = DCS_array[subject_index[0][0], 8]
subject_h_y_before = DCS_array[subject_index[0][0], 9]
subject_h_y_after = DCS_array[subject_index[0][0], 10]
subject_remark_incl_excl = DCS_array[subject_index[0][0], 11]

subject_remark_procedure = DCS_array[subject_index[0][0], 12]
subject_rigidity = DCS_array[subject_index[0][0], 13]
subject_remark_ET_diagnosis = DCS_array[subject_index[0][0], 14]
subject_motor_symptoms = DCS_array[subject_index[0][0], 15]
subject_alcohol_on_tremor = DCS_array[subject_index[0][0], 16]

subject_fam_hist = DCS_array[subject_index[0][0], 17]
subject_onset_symptoms = DCS_array[subject_index[0][0], 18]
subject_duration_onset = DCS_array[subject_index[0][0], 19]
subject_diag_year = DCS_array[subject_index[0][0], 20]
subject_diag_duration = DCS_array[subject_index[0][0], 21]

subject_tremor_more = DCS_array[subject_index[0][0], 22]
subject_brady_more = DCS_array[subject_index[0][0], 23]
subject_last_dose = DCS_array[subject_index[0][0], 24]
subject_participation_time = DCS_array[subject_index[0][0], 25]
subject_duration_no_med = DCS_array[subject_index[0][0], 26]

subject_prescribed_med = DCS_array[subject_index[0][0], 27]
subject_intake_freq = DCS_array[subject_index[0][0], 28]
subject_left_upper = DCS_array[subject_index[0][0], 29]
subject_left_fore = DCS_array[subject_index[0][0], 30]
subject_left_hand = DCS_array[subject_index[0][0], 31]

subject_right_upper = DCS_array[subject_index[0][0], 32]
subject_right_fore = DCS_array[subject_index[0][0], 33]
subject_right_hand = DCS_array[subject_index[0][0], 34]

subject_info_dataframe = pd.DataFrame(columns = ['', ''])

subject_info_dataframe.loc[0] = ["Subject Group", subject_group]
subject_info_dataframe.loc[1] = ["Subject Code", subject_code]
subject_info_dataframe.loc[2] = ["Study Site", subject_study_site]
subject_info_dataframe.loc[3] = ["Age", subject_age]
subject_info_dataframe.loc[4] = ["Gender", subject_gender]

subject_info_dataframe.loc[5] = ["Race", subject_race]
subject_info_dataframe.loc[6] = ["Height (cm)", subject_height]
subject_info_dataframe.loc[7] = ["Weight (kg)", subject_weight]
subject_info_dataframe.loc[8] = ["Dominant Hand", subject_dominant_hand]
subject_info_dataframe.loc[9] = ["H&Y stage (before)", subject_h_y_before]

subject_info_dataframe.loc[10] = ["H&Y stage (after)", subject_h_y_after]         
subject_info_dataframe.loc[11] = ["Remark on Inclusion & Exclusion Criteria", subject_remark_incl_excl]
subject_info_dataframe.loc[12] = ["Remark on Clinical Study Procedure", subject_remark_procedure]
subject_info_dataframe.loc[13] = ["Rigidity Test", subject_rigidity]
subject_info_dataframe.loc[14] = ["Remark on Diagnosis of ET", subject_remark_ET_diagnosis]

subject_info_dataframe.loc[15] = ["Motor Symptoms", subject_motor_symptoms]        
subject_info_dataframe.loc[16] = ["Tremor's Response on Intake of Alcohol", subject_alcohol_on_tremor]
subject_info_dataframe.loc[17] = ["Family History", subject_fam_hist]
subject_info_dataframe.loc[18] = ["Onset of Motor Symptoms", subject_onset_symptoms]
subject_info_dataframe.loc[19] = ["Duration Since Onset of Motor Symptoms", subject_duration_onset]

subject_info_dataframe.loc[20] = ["Year of Diagnosis", subject_diag_year]
subject_info_dataframe.loc[21] = ["Duration Since Diagnosis", subject_diag_duration]
subject_info_dataframe.loc[22] = ["Tremor More Affected Hand", subject_tremor_more]
subject_info_dataframe.loc[23] = ["Bradykinesia More Affected Hand", subject_brady_more]
subject_info_dataframe.loc[24] = ["Last Dose", subject_last_dose]

subject_info_dataframe.loc[25] = ["Study Start Time", subject_participation_time]
subject_info_dataframe.loc[26] = ["Duration between Last Dose and Study Start Time", subject_duration_no_med]
subject_info_dataframe.loc[27] = ["Prescribed Medications", subject_prescribed_med]
subject_info_dataframe.loc[28] = ["Medications Intake Frequency", subject_intake_freq]
subject_info_dataframe.loc[29] = ["Length of Upper Arm (L)", subject_left_upper]

subject_info_dataframe.loc[30] = ["Length of Forearm (L)", subject_left_fore]
subject_info_dataframe.loc[31] = ["Length of Hand (L)", subject_left_hand]
subject_info_dataframe.loc[32] = ["Length of Upper Arm (R)", subject_right_upper]
subject_info_dataframe.loc[33] = ["Length of Forearm (R)", subject_right_fore]
subject_info_dataframe.loc[34] = ["Length of Hand (R)", subject_right_hand]

print(f'Size of subject info dataframes (DCS): {np.shape(subject_info_dataframe)}')

# Path directory for the new excel file
new_excel_folder_path = f'C:/Users/adria/OneDrive - Monash University/2 Raw Combined Data/'#{subject_group}/'
# new_excel_folder_path = 'C:/Users/adria/OneDrive - Monash University/MEngSc/Measurement System/1 Validation/Starting Position - Movement/'

# Create the output folder if it doesn't exist
if not os.path.exists(new_excel_folder_path):
    os.makedirs(new_excel_folder_path)

new_excel_file_path = new_excel_folder_path + f'{subject_code}.xlsx'

with pd.ExcelWriter(new_excel_file_path) as writer:
    subject_info_dataframe.to_excel(writer, sheet_name='Data Collection Sheet', header = True)
    h5_all_dataframe.to_excel(writer, sheet_name='Motion Data', header = True)
    csv_all_dataframe.to_excel(writer, sheet_name='Joint Angles', header = True)

end_time = time.time()
elapsed_time = end_time - start_time
if (elapsed_time >= 60):
    elapsed_minute = int(np.floor(elapsed_time/60))
    elapsed_second = int(np.ceil(elapsed_time - elapsed_minute*60))
    if (elapsed_minute > 1):
        print(f'{subject_code} excel file has been successfully created, which took {elapsed_minute} minutes {elapsed_second} seconds.\n')
    else:
        print(f'{subject_code} excel file has been successfully created, which took {elapsed_minute} minute {elapsed_second} seconds.\n')
else:
    elapsed_time = int(np.ceil(elapsed_time))
    print(f'{subject_code} excel file has been successfully created, which took {elapsed_time} seconds.\n')


Subject Code: HC 68
File name 1: 01_20241203-144208SGT_Free Form_Trial_Joint_Angles.csv
File name 2: 02_20241203-144311SGT_Free Form_Trial_Joint_Angles.csv
File name 3: 03_20241203-144725SGT_Free Form_Trial_Joint_Angles.csv
File name 4: 04_20241203-144510SGT_Free Form_Trial_Joint_Angles.csv
File name 5: 05_20241203-144918SGT_Free Form_Trial_Joint_Angles.csv
File name 6: 06_20241203-145041SGT_Free Form_Trial_Joint_Angles.csv
File name 7: 07_20241203-144614SGT_Free Form_Trial_Joint_Angles.csv
File name 8: 08_20241203-145357SGT_Free Form_Trial_Joint_Angles.csv
File name 9: 09_20241203-150056SGT_Free Form_Trial_Joint_Angles.csv
File name 10: 10_20241203-150421SGT_Free Form_Trial_Joint_Angles.csv
File name 11: 11_20241203-150214SGT_Free Form_Trial_Joint_Angles.csv
File name 12: 12_20241203-150508SGT_Free Form_Trial_Joint_Angles.csv
File name 13: 13_20241203-150629SGT_Free Form_Trial_Joint_Angles.csv
File name 14: 14_20241203-150301SGT_Free Form_Trial_Joint_Angles.csv
File name 15: 01_202412